# Rate- vs Yield-Strategist competition: reproducing Cockx et al. 2024, Fig. 5

_Investigation `rate-yield-strategist-competition` — coder reproduction notebook._

**Question.** Cockx et al. 2024 (PLOS Comp Biol, "Is it selfish to be filamentous in
biofilms?") introduce iDynoMiCS 2.0 and demonstrate it on a classic
microbial-ecology question: does a Rate Strategist (faster growth, lower
yield) or a Yield Strategist (slower growth, higher/more efficient yield)
win a head-to-head competition for a single limiting substrate — and does
the answer depend on how densely the two strategies are seeded? Can
viva-biofilm's Rust + process-bigraph engine, extended with per-species
kinetics and a distributed multi-strategy spawner, reproduce that
competition and its density-dependence?

An investigation demonstrating viva-biofilm's multi-strategy competition
capability against a specific, citable published result: Cockx et al.
2024's Figure 5, where two bacterial growth strategies (Rate vs Yield)
compete over oxygen and the winner flips with seeding density. The
engine's per-species kinetics, distributed alternating spawner, and
per-strategy population/biomass readbacks (added specifically for this
reproduction) are exercised end-to-end using the paper's own Table-K
Monod parameters.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-biofilm/viva-biofilm').is_dir():
    REPO = Path('/home/runner/work/viva-biofilm/viva-biofilm')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_biofilm.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

## Study: `fig5-rs-ys-competition`

**Question.** Cockx et al. 2024 (Fig. 5) report that when a Rate Strategist (RS — faster
growth, lower yield) competes against a Yield Strategist (YS — slower
growth, higher/more efficient yield) for a single shared oxygen substrate
in a 2D biofilm, the winner depends on seeding density: YS is favored at
low density, RS at intermediate density, YS favored again at high density.
Does viva-biofilm's multi-strategy competition engine — seeded with the
paper's exact Table-K RS/YS kinetics — reproduce a density-dependent
competition outcome, and in what direction?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `fig5-rs-ys-competition` | `fig5-rs-ys-competition` | 22 | — |
| `fig5-rs-ys-competition` | `fig5-rs-ys-competition` | 22 | — |
| `fig5-rs-ys-competition` | `fig5-rs-ys-competition` | 22 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `fig5-rs-ys-competition`** — `spec_fig5_rs_ys_competition` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_fig5_rs_ys_competition = load_spec(REPO / 'viva_biofilm/composites/fig5-rs-ys-competition.composite.yaml')
describe_spec(spec_fig5_rs_ys_competition)

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig5-rs-ys-competition ===
STUDY = 'fig5-rs-ys-competition'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Runtime knobs — edit freely. STEPS = number of composite steps;
# INTERVAL = global dt filling ${interval} placeholders (a per-process
# interval pinned in the edit cell above takes precedence).
STEPS_fig5_rs_ys_competition = 22
INTERVAL_fig5_rs_ys_competition = 0.1
STEPS_fig5_rs_ys_competition = 22
INTERVAL_fig5_rs_ys_competition = 0.1
STEPS_fig5_rs_ys_competition = 22
INTERVAL_fig5_rs_ys_competition = 0.1

if RERUN:
    with quiet():  # the sim prints per-step progress; keep it out of the notebook
        # Generic process-bigraph protocol (no workspace runner detected):
        from viva_superpowers.composite_spec import build_composite_from_spec
        comp = build_composite_from_spec(spec_fig5_rs_ys_competition, {'interval': INTERVAL_fig5_rs_ys_competition}, core=core)
        comp.run(STEPS_fig5_rs_ys_competition)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_fig5_rs_ys_competition, {'interval': INTERVAL_fig5_rs_ys_competition}, core=core)
        comp.run(STEPS_fig5_rs_ys_competition)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_fig5_rs_ys_competition, {'interval': INTERVAL_fig5_rs_ys_competition}, core=core)
        comp.run(STEPS_fig5_rs_ys_competition)  # writes the composite's declared emitter
    print(f'ran 3 simulation(s) -> {RUNS_DB}')
else:
    print("RERUN=False — rendering committed", RUNS_DB)

### Visualizations

_Results are shown by the figures below, produced by the run above._


**outcome vs seeding density**


In [ ]:
# outcome vs seeding density
show_viz(_render_one('image:charts/outcome_vs_density.png', {'title': 'Competition outcome vs seeding density', 'caption': 'Final RS biomass fraction at n_each=5/10/50, with the 0.5 tie line marked'}, RUNS_DB, STUDY_YAML))

**colony at n_each=5 (Fig. 5 view)**


In [ ]:
# colony at n_each=5 (Fig. 5 view)
show_viz(_render_one('image:charts/colony_density5.png', {'title': 'Cockx 2024 Fig. 5 view — colony at n_each=5 (t=21d)', 'caption': 'Day-21 biofilm cross-section, cells colored by strategy (RS blue, YS red) over the oxygen field (grayscale: dark=depleted near substratum, light=bulk O2), a 60-µm interior window. Low-density seeding — YS ahead.'}, RUNS_DB, STUDY_YAML))

**colony at n_each=10 (Fig. 5 view)**


In [ ]:
# colony at n_each=10 (Fig. 5 view)
show_viz(_render_one('image:charts/colony_density10.png', {'title': 'Cockx 2024 Fig. 5 view — colony at n_each=10 (t=21d)', 'caption': 'Same view at intermediate-density seeding — RS ahead. Cells over the oxygen gradient they draw down.'}, RUNS_DB, STUDY_YAML))

**colony at n_each=50 (Fig. 5 view)**


In [ ]:
# colony at n_each=50 (Fig. 5 view)
show_viz(_render_one('image:charts/colony_density50.png', {'title': 'Cockx 2024 Fig. 5 view — colony at n_each=50 (t=21d)', 'caption': 'Same view at high-density seeding — RS ahead. Cells colored by strategy over the oxygen field.'}, RUNS_DB, STUDY_YAML))

**RS fraction over time**


In [ ]:
# RS fraction over time
show_viz(_render_one('image:charts/fraction_over_time.png', {'title': 'RS biomass fraction over time', 'caption': 'RS/(RS+YS) biomass fraction vs time, one line per seeding density'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| density-dependent-competition | kind=report_card_axis card=workspace/studies/fig5-rs-ys-competition/viz/report_card group=competition | op verdict_at_least level within_tol |
| density-changes-the-winner | kind=report_card_axis card=workspace/studies/fig5-rs-ys-competition/viz/report_card group=winner-reversal | op verdict_at_least level within_tol |
| paper-flip-fidelity | kind=report_card_axis card=workspace/studies/fig5-rs-ys-competition/viz/report_card group=exact-flip | op verdict_at_least level within_tol |
